# Processing Allen Brain Observatory Data with neuros-astro

This notebook shows how to run the neuros-astro pipeline on your existing
Allen Visual Coding 2-photon calcium imaging data.

**Your data location:**
`packages/neuros-mechint/examples/allen_data_demo/data/2p_sessions/`

**Compute Requirements**: CPU only! ✅

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# neuros-astro
from neuros_astro.io.allen_loader import (
    load_allen_session_from_npz,
    convert_trial_aligned_to_continuous,
)
from neuros_astro.events.event_detection import detect_events_from_traces
from neuros_astro.networks.functional_connectivity import build_event_coactivation_graph
from neuros_astro.tokenization.event_tokenizer import AstroEventTokenizer
from neuros_astro.visualization.event_plots import (
    plot_event_raster,
    plot_event_distributions,
    plot_event_statistics_summary,
    plot_network_graph,
)
from neuros_astro.export.to_neurofm import (
    save_tokenized_sequence_npz,
    build_neurofm_manifest,
    save_neurofm_manifest,
)

print("✓ Imports successful!")

## Step 1: Load Your Allen Data

You have preprocessed Allen 2P sessions already!
Let's load one and inspect it.

In [ ]:
# Path to your Allen data
allen_data_dir = Path("../../neuros-mechint/examples/allen_data_demo/data/2p_sessions")

# List available sessions
sessions = sorted(allen_data_dir.glob("*.npz"))
print(f"📂 Found {len(sessions)} Allen 2P sessions:\n")
for i, session in enumerate(sessions[:5], 1):
    size_mb = session.stat().st_size / 1024 / 1024
    print(f"  {i}. {session.name} ({size_mb:.2f} MB)")

if len(sessions) > 5:
    print(f"  ... and {len(sessions) - 5} more")

In [ ]:
# Load first session
session_path = sessions[0]
print(f"\n🔬 Loading: {session_path.name}")

trial_responses, metadata = load_allen_session_from_npz(session_path)

print(f"\n✓ Loaded successfully!")
print(f"  Shape: {trial_responses.shape} [trials, cells]")
print(f"  Trials: {metadata['n_trials']}")
print(f"  Cells: {metadata['n_cells']}")
print(f"\nMetadata keys: {list(metadata.keys())}")

## Step 2: Convert to Continuous Traces

Your data is trial-aligned. We'll convert it to pseudo-continuous traces
for event detection.

**Note:** This is an approximation. Ideally, use continuous dF/F from AllenSDK,
but this works for demonstration!

In [ ]:
# Convert to continuous (pseudo-continuous from trials)
trial_duration_s = 0.5  # Typical trial window
frame_rate_hz = 30.0  # Allen 2P imaging rate

print("🔄 Converting trial-aligned data to continuous traces...")
traces, timestamps = convert_trial_aligned_to_continuous(
    trial_responses=trial_responses,
    trial_duration_s=trial_duration_s,
    frame_rate_hz=frame_rate_hz,
)

print(f"✓ Continuous traces: {traces.shape} [cells, timepoints]")
print(f"✓ Duration: {timestamps[-1] - timestamps[0]:.1f}s")
print(f"✓ Frame rate: {frame_rate_hz} Hz")

print("\n⚠️  Note: This is pseudo-continuous from trial data.")
print("   For best results, use continuous dF/F from AllenSDK.")

In [ ]:
# Visualize a few cells
fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)

for i in range(5):
    axes[i].plot(timestamps, traces[i], 'k-', linewidth=0.5, alpha=0.7)
    axes[i].set_ylabel(f'Cell {i}\nΔF/F', fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

axes[-1].set_xlabel('Time (s)', fontsize=12)
axes[0].set_title('Allen 2P Calcium Traces (First 5 Cells)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Notice the trial-aligned structure")
print("✓ Looking for cells with slow, sustained events...")

## Step 3: Detect Events

Run neuros-astro event detection on these traces.

**Important:** Without genetic labeling, we can't definitively say which cells
are astrocytes! This is exploratory analysis to see if we can find
slow-timescale calcium events.

In [ ]:
# Detect events
session_id = session_path.stem

print(f"🔍 Detecting calcium events in {metadata['n_cells']} cells...")
print("   (This may take 10-30 seconds)\n")

events = detect_events_from_traces(
    traces=traces,
    frame_rate_hz=frame_rate_hz,
    session_id=session_id,
    z_threshold=2.0,  # Adjust based on noise
    min_duration_s=1.0,  # Look for slow events (astrocyte-like)
    merge_gap_s=0.5,
)

print(f"✓ Detected {len(events)} events across {metadata['n_cells']} cells")
print(f"✓ Average: {len(events)/metadata['n_cells']:.1f} events per cell")

if len(events) > 0:
    plot_event_statistics_summary(events)

In [ ]:
# Visualize event raster
if len(events) > 0:
    fig, ax = plot_event_raster(
        events,
        frame_rate_hz=frame_rate_hz,
        figsize=(14, 8),
        title=f"Allen 2P Events: {session_path.name}"
    )
    plt.show()
else:
    print("⚠️  No events detected. Try adjusting z_threshold or min_duration_s")

In [ ]:
# Feature distributions
if len(events) > 0:
    fig = plot_event_distributions(events, figsize=(14, 10))
    plt.show()
    
    print("\n📊 Interpretation:")
    durations = np.array([e.duration_s for e in events])
    median_duration = np.median(durations)
    
    if median_duration > 2.0:
        print(f"  ✓ Median duration {median_duration:.2f}s suggests slow events")
        print("    (Compatible with astrocyte timescales)")
    else:
        print(f"  ⚠️  Median duration {median_duration:.2f}s is fast")
        print("    (More neuron-like, but could still be interesting!)")

## Step 4: Build Networks

Construct functional connectivity based on event coactivation.

In [ ]:
if len(events) > 10:  # Need sufficient events
    print("🕸️  Building coactivation networks...")
    graphs = build_event_coactivation_graph(
        events=events,
        session_id=session_id,
        frame_rate_hz=frame_rate_hz,
        window_size_s=5.0,  # Shorter windows for trial data
        stride_s=2.0,
        min_edge_weight=0.2,
    )
    
    print(f"✓ Generated {len(graphs)} network windows")
    
    if len(graphs) > 0:
        # Plot first network
        fig, ax = plot_network_graph(
            graphs[0],
            figsize=(10, 10),
            layout="spring",
            show_weights=True
        )
        plt.show()
        
        print("\n✓ Network shows functional connectivity")
        print("✓ Nodes = cells, Edges = coactivation")
else:
    print(f"⚠️  Only {len(events)} events detected")
    print("   Need more events for meaningful networks")
    graphs = []

## Step 5: Tokenize & Export

Create model-ready tokens for neuroFMx integration.

In [ ]:
if len(events) > 0:
    # Tokenize
    print("🎯 Tokenizing events...")
    tokenizer = AstroEventTokenizer(normalize=True)
    tokens = tokenizer.tokenize(events, session_id=session_id)
    
    print(f"✓ Generated {len(tokens.tokens)} token vectors")
    print(f"✓ Features: {', '.join(tokens.feature_names[:5])}...")
    
    # Save
    output_dir = Path("./allen_outputs")
    output_dir.mkdir(exist_ok=True)
    
    tokens_path = output_dir / f"{session_id}_astro_tokens.npz"
    save_tokenized_sequence_npz(tokens, tokens_path)
    print(f"\n✓ Tokens saved: {tokens_path}")
    
    # Create manifest
    manifest = build_neurofm_manifest(
        session_id=session_id,
        modalities={
            "astro": {
                "type": "event_tokens",
                "path": tokens_path.name,
                "sampling": "irregular",
            }
        },
        metadata={
            "source": "Allen Brain Observatory",
            "n_events": len(events),
            "n_cells": metadata['n_cells'],
        }
    )
    
    manifest_path = output_dir / f"{session_id}_manifest.json"
    save_neurofm_manifest(manifest, manifest_path)
    print(f"✓ Manifest saved: {manifest_path}")
    
    print("\n🎉 Ready for neuroFMx integration!")

## Validation & Next Steps

### What We Learned:
1. ✅ neuros-astro pipeline works on real Allen data
2. ✅ Can detect slow calcium events
3. ✅ Can build coactivation networks
4. ✅ Tokens are ready for foundation models

### Important Caveats:
- ⚠️  **No astrocyte labels**: We don't know which cells are actually astrocytes!
- ⚠️  **Trial-aligned data**: Pseudo-continuous traces are approximations
- ⚠️  **Exploratory**: This is initial validation, not definitive science

### For Publication-Quality Work:
1. **Use continuous dF/F**: Load from AllenSDK directly
2. **Find astrocyte-labeled datasets**: DANDI may have GFAP-labeled sessions
3. **Validate biology**: Check if event properties match literature
4. **Compare to baselines**: What do neurons look like with same detection?

### Next Steps:
- Process all sessions in `2p_sessions/`
- Aggregate statistics across sessions
- Compare to synthetic data validation
- Move to neuroFMx integration (notebook 03)

## Batch Process All Sessions (Optional)

Process all your Allen sessions at once.

In [ ]:
# Uncomment to process all sessions
# results = []
# 
# for session_path in sessions:
#     try:
#         print(f"\nProcessing {session_path.name}...")
#         
#         # Load
#         trial_data, meta = load_allen_session_from_npz(session_path)
#         traces, time = convert_trial_aligned_to_continuous(trial_data)
#         
#         # Detect events
#         events = detect_events_from_traces(
#             traces, frame_rate_hz=30.0, session_id=session_path.stem,
#             z_threshold=2.0, min_duration_s=1.0
#         )
#         
#         # Store results
#         results.append({
#             'session_id': session_path.stem,
#             'n_cells': meta['n_cells'],
#             'n_events': len(events),
#             'events_per_cell': len(events) / meta['n_cells'] if meta['n_cells'] > 0 else 0,
#         })
#         
#         print(f"  ✓ {len(events)} events detected")
#         
#     except Exception as e:
#         print(f"  ✗ Error: {e}")
# 
# # Summary
# import pandas as pd
# df = pd.DataFrame(results)
# print("\n" + "="*60)
# print("BATCH PROCESSING SUMMARY")
# print("="*60)
# print(df.to_string())
# print(f"\nTotal events: {df['n_events'].sum()}")
# print(f"Mean events per cell: {df['events_per_cell'].mean():.2f}")